# SFT CustomLLM on a Colab GPU

This notebook downloads `HuggingFaceH4/ultrachat_200k`, formats user/assistant conversations with ChatML control tokens, trains the larger `CustomLLM` for 12 SFT epochs with CUDA AMP, and downloads `llm_export_bundle.zip` containing the best checkpoint and tokenizer.

Select a GPU runtime in **Runtime > Change runtime type** before starting.

In [ ]:
!pip -q install torch tokenizers datasets

import os
import sys
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU runtime in Colab before training.')
print('GPU:', torch.cuda.get_device_name(0))

## Get the repository

Set `REPO_URL` to the GitHub repository URL. If it is left empty, the next cell opens a Colab upload dialog for a ZIP containing the repository files.

In [ ]:
from pathlib import Path
import shutil

REPO_URL = 'https://github.com/Narco25/test_llm.git'  # Example: 'https://github.com/your-user/test_llm.git'
WORKSPACE = Path('/content/test_llm')

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)

if REPO_URL:
    !git clone {REPO_URL} {WORKSPACE}
else:
    from google.colab import files
    print('Upload a ZIP of the repository, then rerun this cell if needed.')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
    if zip_name is None:
        raise ValueError('Upload a repository ZIP or set REPO_URL.')
    import zipfile
    with zipfile.ZipFile(zip_name) as archive:
        archive.extractall('/content')
    extracted = Path('/content') / Path(zip_name).stem
    if extracted.exists() and extracted != WORKSPACE:
        extracted.rename(WORKSPACE)

os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
print('Workspace:', Path.cwd())

In [ ]:
from pathlib import Path
from model.bpe_data import load_instruction_conversations, save_formatted_conversations

# Download and normalize a standard instruction/chat dataset.
INSTRUCTION_DATASET = 'HuggingFaceH4/ultrachat_200k'
INSTRUCTION_SPLIT = 'train_sft'
MAX_EXAMPLES = 20_000
conversations = load_instruction_conversations(
    INSTRUCTION_DATASET,
    split=INSTRUCTION_SPLIT,
    max_examples=MAX_EXAMPLES,
)
formatted_data_path = save_formatted_conversations(
    conversations,
    'model/instruction_data.txt',
)
print('Instruction conversations:', len(conversations))
print('Formatted data:', formatted_data_path)

In [ ]:
from model.bpe_data import train_or_load_tokenizer

# Retrain the tokenizer on ChatML-formatted instruction data.
tokenizer_path = Path('model/tokenizer.json')
if tokenizer_path.exists():
    tokenizer_path.unlink()
tokenizer = train_or_load_tokenizer(
    formatted_data_path,
    tokenizer_path,
    vocab_size=4096,
)
print('BPE vocabulary size:', tokenizer.get_vocab_size())
for token in ('<PAD>', '<EOS>', '<|im_start|>', '<|im_end|>'):
    print(token, '->', tokenizer.token_to_id(token))

## Supervised fine-tuning

This cell trains on ChatML-formatted instruction conversations with assistant-only loss, `d_model=384`, `n_layers=6`, `n_heads=6`, `block_size=512`, AdamW, linear warmup, and cosine decay. It saves the best validation checkpoint to `model/weights/best_model.pt`.

In [ ]:
from pathlib import Path
from torch import nn
from model.architecture import CustomLLM
from model.bpe_data import InstructionDataset, create_dataloader
from model.train import create_optimizer, create_warmup_cosine_scheduler

DEVICE = torch.device('cuda')
EPOCHS = 12
BATCH_SIZE = 32
SEQUENCE_LENGTH = 512
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1


dataset = InstructionDataset(
    conversations,
    tokenizer,
    sequence_length=SEQUENCE_LENGTH,
)
if len(dataset) < 2:
    raise ValueError('The instruction dataset must provide at least two trainable windows.')
validation_size = max(1, int(len(dataset) * 0.02))
training_size = len(dataset) - validation_size
training_dataset, validation_dataset = torch.utils.data.random_split(
    dataset, [training_size, validation_size],
    generator=torch.Generator().manual_seed(42),
)
training_loader = create_dataloader(training_dataset, BATCH_SIZE, shuffle=True)
validation_loader = create_dataloader(validation_dataset, BATCH_SIZE, shuffle=False)

model = CustomLLM(
    tokenizer=tokenizer,
    vocab_size=tokenizer.get_vocab_size(),
    d_model=384,
    n_heads=6,
    n_layers=6,
    block_size=512,
).to(DEVICE)

optimizer = create_optimizer(model, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * max(1, len(training_loader))
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = create_warmup_cosine_scheduler(optimizer, total_steps, warmup_steps)
loss_fn = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=True)
best_validation_loss = float('inf')
checkpoint_path = Path('model/weights/best_model.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    train_loss_total = 0.0
    for batch_index, batch in enumerate(training_loader, start=1):
        input_ids = batch['input_ids'].to(DEVICE)
        targets = batch['labels'].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(input_ids)
            loss = loss_fn(logits.view(-1, tokenizer.get_vocab_size()), targets.view(-1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss_total += loss.item()
        if batch_index % 100 == 0:
            print(f'epoch {epoch + 1}/{EPOCHS}, batch {batch_index}/{len(training_loader)}, loss={loss.item():.4f}', flush=True)

    model.eval()
    validation_loss_total = 0.0
    with torch.no_grad():
        for batch in validation_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            targets = batch['labels'].to(DEVICE)
            with torch.amp.autocast('cuda'):
                logits = model(input_ids)
                validation_loss_total += loss_fn(
                    logits.view(-1, tokenizer.get_vocab_size()), targets.view(-1)
                ).item()
    train_loss = train_loss_total / max(1, len(training_loader))
    validation_loss = validation_loss_total / max(1, len(validation_loader))
    print(f'epoch {epoch + 1}/{EPOCHS}, train_loss={train_loss:.4f}, validation_loss={validation_loss:.4f}', flush=True)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'validation_loss': validation_loss,
            'epoch': epoch + 1,
        }, checkpoint_path)
        print('saved best checkpoint to', checkpoint_path, flush=True)

print('Best validation loss:', best_validation_loss)
print('Checkpoint exists:', checkpoint_path.exists())

In [ ]:
# Bundle the best checkpoint and tokenizer for local download.
from google.colab import files
import zipfile

export_bundle_path = Path('llm_export_bundle.zip')
if not checkpoint_path.exists():
    raise FileNotFoundError(f'Checkpoint was not created: {checkpoint_path}')
if not Path('model/tokenizer.json').exists():
    raise FileNotFoundError('Tokenizer was not created: model/tokenizer.json')

with zipfile.ZipFile(export_bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(checkpoint_path, arcname='model/weights/best_model.pt')
    archive.write('model/tokenizer.json', arcname='model/tokenizer.json')

print('Created export bundle:', export_bundle_path)
files.download(str(export_bundle_path))